# More than two labels

A basic SVM compares two classes. With three classes, train three machines. Each machine defends one class against the other two. This is one-versus-rest.

Use one point from each class:

$$
A = (0, 0), \qquad B = (2, 0), \qquad C = (0, 2)
$$

A new point is given to all three machines. The machine with the largest score names the class. If two scores tie for the lead, say so. Do not break the tie in silence.


## $A$ against the rest

$A$ is the positive point. $B$ and $C$ are negative. Both negative points lie at distance $2$ from the origin, and the segment between them is the line $x_1 + x_2 = 2$. The closest point on that segment to the origin is $(1, 1)$. The halfway line is

$$
x_1 + x_2 = 1
$$

Scaled to functional margin $1$, the score is

$$
f_A(x_1, x_2) = 1 - x_1 - x_2
$$

so $\mathbf{w}_A = (-1, -1)$ and $b_A = 1$. Check:

$$
\begin{aligned}
A &: 1, && \text{margin } 1 \\
B &: -1, && \text{margin } 1 \\
C &: -1, && \text{margin } 1
\end{aligned}
$$

The multipliers are $\alpha_A = 1$, $\alpha_B = 1/2$, $\alpha_C = 1/2$. Their signed sum is $1 - 1/2 - 1/2 = 0$, and they rebuild $\mathbf{w}_A$.


## $B$ against the rest, and $C$ against the rest

Both non-$B$ points have first coordinate $0$. The margin edge through them is the line $x_1 = 0$, and the positive edge through $B$ is $x_1 = 2$. The center is $x_1 = 1$:

$$
f_B(x_1, x_2) = x_1 - 1
$$

By the same geometry with the axes swapped,

$$
f_C(x_1, x_2) = x_2 - 1
$$

$C$ lies on the edge of $f_B$, because $f_B(0, 2) = -1$. Its multiplier in the $B$-versus-rest problem is nevertheless $0$. A point may sit on the edge without being the one that pins the weight. Here $A$ and $B$ already determine $\mathbf{w}_B = (1, 0)$.


In [1]:
# Three hard-margin scores. The training points vote for their own class.
from fractions import Fraction

def fA(x1, x2):
    return 1 - x1 - x2

def fB(x1, x2):
    return x1 - 1

def fC(x1, x2):
    return x2 - 1

def winners(x1, x2):
    scores = {"A": fA(x1, x2), "B": fB(x1, x2), "C": fC(x1, x2)}
    best = max(scores.values())
    return sorted(name for name, value in scores.items() if value == best)

assert fA(0, 0) == 1 and fB(0, 0) == -1 and fC(0, 0) == -1
assert winners(0, 0) == ["A"]
assert winners(2, 0) == ["B"]
assert winners(0, 2) == ["C"]
# (1, 1) ties B and C at score 0. A is strictly behind.
assert fA(1, 1) == -1 and fB(1, 1) == 0 and fC(1, 1) == 0
assert winners(1, 1) == ["B", "C"]
assert winners(3, 0) == ["B"]

# Multipliers for A versus rest rebuild w = (-1, -1).
alpha = {"A": Fraction(1), "B": Fraction(1, 2), "C": Fraction(1, 2)}
labels = {"A": 1, "B": -1, "C": -1}
points = {"A": (0, 0), "B": (2, 0), "C": (0, 2)}
w1 = sum(alpha[k] * labels[k] * points[k][0] for k in alpha)
w2 = sum(alpha[k] * labels[k] * points[k][1] for k in alpha)
assert (w1, w2) == (-1, -1)
print("winners at (1, 1)", winners(1, 1))
print("winners at (3, 0)", winners(3, 0))


winners at (1, 1) ['B', 'C']
winners at (3, 0) ['B']


## Why one-versus-rest on a line can fail

If the classes sit at positions $0$, $3$, and $6$ on one line, the middle class has points of the other classes on both sides. No single threshold separates "middle" from "the rest." One-versus-rest is not guaranteed to find a line for every class. The triangle above works because each vertex can face the other two.


## A pitfall

The three scores do not sum to $1$ and they are not probabilities. $f_B(3, 0) = 2$ means the point is far on the positive side of the $B$ line. It does not mean "probability $2$."

## What to carry forward

Train one hard-margin machine per class. Here $f_A = 1 - x_1 - x_2$, $f_B = x_1 - 1$, and $f_C = x_2 - 1$. The point $(1, 1)$ is a tie between $B$ and $C$.
